# 👁️ Smart Eye — Train model nhận diện có thêm cầu thang, cột điện, ổ gà...

Notebook này chạy trên **Google Colab** (miễn phí, có GPU, chạy Linux nên export TFLite được — **không cần WSL/Mac**).

**Quy trình:** tải 1 phần COCO (giữ người, xe...) → thêm dataset Roboflow / Mendeley / Kaggle + ảnh nhóm tự gán nhãn → gộp theo
`training/classes.yaml` → train YOLO → kiểm tra → xuất `smart_eye.tflite` cho app.

**Trước khi chạy:** *Runtime → Change runtime type → **T4 GPU***. Toàn bộ mất khoảng 1,5–3 giờ (phần lớn là train).

Kết quả và dataset lưu trên **Google Drive** (`MyDrive/smart_eye`) để Colab ngắt kết nối vẫn không mất.

## 1. Chuẩn bị

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install ultralytics fiftyone roboflow kaggle ai-edge-litert pyyaml

In [ ]:
REPO_BRANCH = 'feature/perf-and-classes'  # nhánh chứa thư mục training/ (đổi thành 'main' sau khi merge)
!rm -rf smart-eye && git clone -q -b $REPO_BRANCH https://github.com/langvietthanh/smart-eye.git
%cd smart-eye

from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/smart_eye'
!mkdir -p $WORK/datasets $WORK/own_data $WORK/raw_photos $WORK/export

## 2. Dữ liệu COCO (giữ các lớp cũ để model không "quên")
Chỉ tải ảnh có người, xe, ghế, biển báo... (~6800 ảnh, vài GB) — **chỉ cần chạy 1 lần**, lần sau đã có trên Drive.

In [ ]:
import os
if not os.path.exists(f'{WORK}/datasets/coco_subset/dataset.yaml'):
    !python training/scripts/fetch_coco_subset.py --out $WORK/datasets/coco_subset --train 6000 --val 800
else:
    print('Đã có COCO subset trên Drive — bỏ qua')

## 3. Dataset có sẵn trên Roboflow Universe
1. Vào https://universe.roboflow.com, tìm: `pothole`, `stairs`, `utility pole`, `railing`, `manhole`, `bollard`,
   `traffic cone`, `curb`, `vietnam traffic sign`, `street vendor`...
2. Chọn dataset **giấy phép cho phép dùng** (VD CC BY 4.0), ảnh giống cảnh đường phố thật, nhiều ảnh.
3. Bấm **Download → YOLOv8 → show download code**, chép 3 giá trị `workspace`, `project`, `version` vào danh sách dưới.
4. API key: biểu tượng 🔑 (Secrets) bên trái Colab → thêm `ROBOFLOW_API_KEY`.

Tên lớp của dataset không cần khớp — `build_dataset.py` tự đổi qua `aliases` trong `classes.yaml`
(lớp lạ sẽ bị bỏ và được liệt kê trong báo cáo để bổ sung alias nếu cần).

In [ ]:
ROBOFLOW = [
    # ('workspace', 'project', version),
]

if ROBOFLOW:
    from google.colab import userdata
    from roboflow import Roboflow
    rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
    for ws, proj, ver in ROBOFLOW:
        dest = f'{WORK}/datasets/rf_{proj}_v{ver}'
        if os.path.exists(dest):
            print('Đã có', dest); continue
        rf.workspace(ws).project(proj).version(ver).download('yolov8', location=dest)

## 3b. Dataset trên Mendeley Data
Điền `(id, phiên bản, tên)` lấy từ link `data.mendeley.com/datasets/<id>/<phiên bản>`.

- **BPID** — 161 ảnh ổ gà ở Bandung (Indonesia), 263 ổ gà, nắng / râm / ướt / ban đêm, giấy phép **CC BY 4.0**
  (phải ghi nguồn). Tác giả thiết kế làm **tập kiểm tra độc lập** → mặc định vào tập **val** (ảnh nằm trong thư mục `test`).
  Muốn dùng để **train**: đặt tên bắt đầu bằng `train_` (VD `'train_bpid'`).

In [ ]:
MENDELEY = [
    ('rgymy6dwdd', 1, 'bpid'),  # Bandung Pothole Image Dataset — DOI 10.17632/rgymy6dwdd.1
]
for ds_id, ver, name in MENDELEY:
    dest = f'{WORK}/datasets/{name}'
    if not os.path.exists(dest):
        !python training/scripts/fetch_mendeley.py $ds_id --version $ver --out "$dest"

## 3c. Dataset trên Kaggle
1. Đăng nhập kaggle.com → ảnh đại diện → *Settings* → mục **API**.
2. Colab → biểu tượng 🔑 (Secrets) bên trái → thêm secret, bật *Notebook access*:
   - Nếu Kaggle cho tải file **`kaggle.json`** (*Create Legacy API Key*): mở file, thêm `KAGGLE_USERNAME` = `username`,
     `KAGGLE_KEY` = `key`.
   - Nếu Kaggle cấp **API token** dạng mới (*Generate New Token*): thêm `KAGGLE_API_TOKEN` = token đó.

- **Pothole Detection** (`andrewmvd/pothole-detection`) — 665 ảnh ổ gà, nhãn PASCAL VOC (XML, `build_dataset.py` tự đổi
  sang YOLO), giấy phép DbCL v1.0, ghi nguồn MakeML. Đặt tên `train_...` → dùng để **train** (BPID ở bước 3b để kiểm tra).

In [ ]:
KAGGLE = [
    ('andrewmvd/pothole-detection', 'train_kaggle_potholes'),
]
if KAGGLE:
    from google.colab import userdata
    for secret in ('KAGGLE_API_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY'):
        try:
            os.environ[secret] = userdata.get(secret)
        except Exception:
            pass  # Chỉ cần 1 trong 2 kiểu: KAGGLE_API_TOKEN, hoặc KAGGLE_USERNAME + KAGGLE_KEY
    for slug, name in KAGGLE:
        dest = f'{WORK}/datasets/{name}'
        if not os.path.exists(dest):
            !kaggle datasets download $slug -p "$dest" --unzip

## 4. Ảnh nhóm tự chụp (quan trọng nhất cho vỉa hè Việt Nam)
**Cách chụp:** điện thoại **dọc, đeo/cầm trước ngực** như khi dùng app, đi bộ trên vỉa hè thật; nhiều giờ trong ngày,
trời nắng/râm/mưa, cả ngày lẫn tối. Quay video rồi trích ảnh 1–2 ảnh/giây cũng được.

**4a. (Tuỳ chọn) Gán nhãn tự động:** chép ảnh vào `MyDrive/smart_eye/raw_photos/<tên đợt>/` rồi chạy ô dưới.
YOLO-World khoanh sẵn box → tải thư mục kết quả về, **sửa lại bằng CVAT / Label Studio / Roboflow**.

**4b. Ảnh đã gán nhãn xong:** export định dạng **YOLO** thành file `.zip`, chép vào `MyDrive/smart_eye/own_data/`.

In [ ]:
# 4a — gán nhãn tự động (bỏ qua nếu không có ảnh mới)
import glob
for batch in sorted(glob.glob(f'{WORK}/raw_photos/*/')):
    name = os.path.basename(batch.rstrip('/'))
    out = f'{WORK}/auto_labeled/{name}'
    if not os.path.exists(out):
        !python training/scripts/auto_label.py --images "$batch" --out "$out"
print('Kết quả gán nhãn tự động ở MyDrive/smart_eye/auto_labeled/ — nhớ kiểm tra & sửa trước khi đưa vào 4b')

In [ ]:
# 4b — giải nén dữ liệu đã gán nhãn xong
import zipfile
for z in sorted(glob.glob(f'{WORK}/own_data/*.zip')):
    dest = f'{WORK}/datasets/own_' + os.path.splitext(os.path.basename(z))[0]
    if not os.path.exists(dest):
        zipfile.ZipFile(z).extractall(dest)
        print('Giải nén', z, '→', dest)

## 5. Gộp dữ liệu theo `classes.yaml`
Đọc kỹ bảng thống kê: lớp có **⚠ thiếu dữ liệu** (< 300 vật) sẽ học kém → bổ sung ảnh cho lớp đó trước khi train lâu.

In [ ]:
def spec(d):
    base = os.path.basename(d.rstrip('/'))
    # Thư mục tên train_* / val_* → ép cả nguồn vào tập đó
    return d + ('#train' if base.startswith('train_') else '#val' if base.startswith('val_') else '')
sources = ' '.join(f'--source "{spec(d)}"' for d in sorted(glob.glob(f'{WORK}/datasets/*/')))
!python training/scripts/build_dataset.py $sources --out /content/smart_eye_ds

## 6. Train
- `yolo11n.pt`: nhỏ, nhanh, hợp điện thoại (có thể đổi `yolov8n.pt`).
- `IMGSZ = 320` khớp app hiện tại; `256` nhanh hơn ~1,5 lần nhưng kém với vật nhỏ ở xa.
- ~7000 ảnh × 80 epoch ≈ 1,5–2 giờ trên T4. **Colab ngắt giữa chừng:** chạy lại bước 1 (chuẩn bị), bước 5 (gộp dữ liệu —
  dataset gộp nằm ở `/content` nên mất khi ngắt), rồi chạy ô dưới với `RESUME = True` — train tiếp từ epoch đang dở.

In [ ]:
from ultralytics import YOLO
BASE, IMGSZ, EPOCHS, BATCH = 'yolo11n.pt', 320, 80, 64
RUN_NAME, RESUME = 'smart_eye_v1', False

if RESUME:
    model = YOLO(f'{WORK}/runs/{RUN_NAME}/weights/last.pt')
    model.train(resume=True)
else:
    model = YOLO(BASE)
    model.train(
        data='/content/smart_eye_ds/data.yaml', imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH,
        patience=20, cos_lr=True, close_mosaic=10,
        project=f'{WORK}/runs', name=RUN_NAME, exist_ok=True,
    )

## 7. Kiểm tra theo từng lớp

In [ ]:
best = YOLO(f'{WORK}/runs/{RUN_NAME}/weights/best.pt')
metrics = best.val(data='/content/smart_eye_ds/data.yaml', imgsz=IMGSZ, plots=False)
print(f'mAP50 chung: {metrics.box.map50:.3f}')
# ap50 xếp theo ap_class_index (chỉ các lớp có trong tập val), KHÔNG theo id lớp
for k, i in enumerate(metrics.box.ap_class_index):
    print(f'  {best.names[int(i)]:<16} mAP50 = {metrics.box.ap50[k]:.3f}')
missing = [n for i, n in best.names.items() if i not in set(int(x) for x in metrics.box.ap_class_index)]
print('Chưa có trong tập val (chưa đánh giá được):', missing)

## 8. Xuất TFLite cho app
Xuất bản **FP32** (bản INT8 hiệu chỉnh ít ảnh bị "chặn trần" điểm tin cậy), rồi chuyển trọng số về định dạng
runtime Android/iOS đọc được, rồi chấm lại đúng như app chạy.

In [ ]:
import shutil
tflite = best.export(format='tflite', imgsz=IMGSZ)
print('Export:', tflite)
!python tool/inline_tflite_buffers.py "$tflite"
!python tool/eval_model.py --model "$tflite" --data /content/smart_eye_ds/data.yaml --conf 0.25,0.35

out = f'{WORK}/export/smart_eye_{RUN_NAME}_{IMGSZ}.tflite'
shutil.copy(tflite, out)
print('Đã lưu:', out)
from google.colab import files
files.download(out)

## 9. Đưa vào app
1. Đổi tên file vừa tải thành **`smart_eye.tflite`**, chép vào `assets/models/` của repo.
2. `flutter run` — app **tự dùng model mới** (tên lớp đọc từ metadata trong model, không cần sửa code).
   Log khởi động phải có: `Model: assets/models/smart_eye.tflite ... nhãn từ metadata`.
3. Thêm / đổi lớp: sửa `training/classes.yaml` **và** `lib/utils/label_catalog.dart` (tên tiếng Việt + nhóm nguy hiểm)
   — `flutter test` sẽ báo nếu 2 file lệch nhau.